In [1]:
print (123)

123


In [2]:
# Load the ground truth questions:

import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")


In [3]:
ground_truth

[{'question': 'Is it okay to join the course late if I just found it now?',
  'document': '74eb249bbf'},
 {'question': 'Can I still take this course even if I missed the start date?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, am I still eligible for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to submit my project before submissions close to get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'I’m a bit late to the course—what do I need to do to still earn the certificate?',
  'document': '74eb249bbf'},
 {'question': 'I registered for the LLM Zoomcamp — when should I expect a confirmation email?',
  'document': '977bf7786c'},
 {'question': 'Do I actually need an acceptance email before I can start the course and hand in homework?',
  'document': '977bf7786c'},
 {'question': 'If I filled out the registration form, does that mean I’m officially on a checked list for the course?',
  'document': '977b

In [5]:
ground_truth [10]

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [6]:
q = ground_truth [10]

In [7]:
q

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [9]:
q['document']

'489dd1c9d9'

In [4]:
# Load the FAQ documents and the search index:

from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

/Users/vincentaina/llm-zoomcamp-evaluation/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [12]:
# Create a lookup table for the original FAQ documents
# We'll use this lookup table to find the original answer for each ground truth question.

doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

# Running RAG

In [13]:
# Import the usual things first

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [14]:
# For this lesson, use RAGWithUsage from the evaluation utilities. It subclasses RAGBase from module 01, so it has the same rag method.

# It stores token usage after each LLM call. Then we can calculate the total cost later.

# It also uses the search boosts we selected in the search tuning lesson: question=1.0, answer=2.0, and section=0.1.

In [17]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [20]:
ground_truth[0]

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

In [19]:
# For each question, RAGBase searches the FAQ, builds a prompt with the retrieved context, and asks the LLM to answer. 
# We save the answer so the next lesson can judge it.

# Run RAG for one question:

rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join late. If you want a certificate, make sure you submit your project while submissions are still open.'

In [23]:
assistant.rag(q['question'])

'Students join the Office Hours or live workshop via **YouTube Live**, not through the Zoom link.\n\nWhat to do:\n- Check the **announcements channel on Telegram or Slack** for the live video URL before it starts.\n- You can also watch on the DataTalksClub **YouTube Channel**.\n- If the session is live, ask questions in **Slido** using the link pinned in chat.\n\nThe Zoom link is only shared with **instructors/presenters/TAs**.'

In [25]:
# Check the cost of this call:
assistant.total_cost()



0.00139875

In [28]:
ground_truth[0]

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

In [29]:
doc_idx

{'74eb249bbf': {'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 '977bf7786c': {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 '489dd1c9d9': {'id': '489dd1c9d9',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours

In [26]:
# Get the original answer from the document ID:

doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [27]:
# Now save both answers in one record:

rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join late. If you want a certificate, make sure you submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

# Processing all questions

In [30]:
# Create a function that processes one ground truth record:

def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [31]:
# Test it on one record
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [34]:
assistant.reset_usage() # Before running the full batch, reset the usage we collected while testing

In [37]:
# Check the cost of this call:
assistant.total_cost()

0.3169635

In [38]:
# This calls the LLM once per ground truth question, so it can take some time. Let's process the questions in parallel and track progress.
# Import the parallel processing helper from the same utility file:

In [39]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [47]:
llm_ids = set(doc_idx.keys())
ground_truth = [rec for rec in ground_truth if rec["document"] in llm_ids]

In [48]:
# Run RAG for all ground truth questions:

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/315 [00:00<?, ?it/s]

In [49]:
# How many ground_truth ids are missing from doc_idx?
missing = [rec["document"] for rec in ground_truth if rec["document"] not in doc_idx]
print(len(missing), missing[:10])

# Sanity check the id type/format
print(type(list(doc_idx.keys())[0]), repr(list(doc_idx.keys())[0]))
print(type(ground_truth[0]["document"]), repr(ground_truth[0]["document"]))

0 []
<class 'str'> '74eb249bbf'
<class 'str'> '74eb249bbf'


In [50]:
doc_idx

{'74eb249bbf': {'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 '977bf7786c': {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 '489dd1c9d9': {'id': '489dd1c9d9',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours

In [43]:
type(doc_idx)

dict

In [51]:
print(len(documents))
print(len(doc_idx))  # should match len(documents) unless there are duplicate ids

# is the id anywhere in the raw documents list, just not in doc_idx?
any(d['id'] == 'c6c2888275' for d in documents)

144
144


False

In [52]:
# is the id anywhere in the raw documents list, just not in doc_idx?
any(d['id'] == 'c6c2888275' for d in documents)

False

In [53]:
# Collect the answer records:


answers = []

for answer_record in results:
    answers.append(answer_record)

In [54]:
# Calculate the total cost:
assistant.total_cost()

0.9488474999999992

In [55]:
#save the answers
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)